# 03 — Image Analysis (EDA)

General exploratory data analysis on preprocessed imagery and labels.
**Assumes `02_image_processing` has been run.**

In [ ]:
import sys
sys.path.append("../")

S2_PATH      = "../data/raw/images/S2H_2023_2023_06_30_nodata_clipped.tif"
CDL_PATH     = "../data/raw/cdl/2023_30m_cdls_10m_clipped.tif"
IMAGE_FOLDER = "../data/raw/images"
CSV_PATH     = "../data/csv/sacramento_2_cdl_labels.csv"
CLASS_IDS    = [1, 3, 24, 54, 69, 75, 76]

---
## Raster Info

In [ ]:
import rasterio

def print_raster_info(path, label=""):
    with rasterio.open(path) as src:
        print(f"── {label} ──────────────────────────")
        print(f"  CRS        : {src.crs}")
        print(f"  Resolution : {src.res}")
        print(f"  Shape      : {src.height} x {src.width}")
        print(f"  Bands      : {src.count}")
        print(f"  Dtype      : {src.dtypes[0]}")
        print(f"  NoData     : {src.nodata}")
        print(f"  Bounds     : {src.bounds}")

print_raster_info(S2_PATH,  "Sentinel-2")
print()
print_raster_info(CDL_PATH, "CDL Label")

---
## CDL Class Distribution

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils.constants import USDA_CDL_NAMES

with rasterio.open(CDL_PATH) as src:
    arr    = src.read(1)
    nodata = src.nodata

unique, counts = np.unique(arr, return_counts=True)
rows = [
    {"class_id": int(c), "class_name": USDA_CDL_NAMES.get(int(c), "Unknown"), "pixel_count": int(n)}
    for c, n in zip(unique, counts)
    if c != nodata and c != 0
]
dist_df = pd.DataFrame(rows).sort_values("pixel_count", ascending=False).reset_index(drop=True)
print(dist_df.to_string(index=False))

In [ ]:
dist_df.sort_values("pixel_count").plot(
    kind="barh", x="class_name", y="pixel_count",
    legend=False, figsize=(10, 6)
)
plt.title("CDL Class Distribution (pixel count)")
plt.xlabel("Pixel Count")
plt.tight_layout()
plt.show()

---
## NDVI — Single Image

In [ ]:
with rasterio.open(S2_PATH) as src:
    red    = src.read(4).astype("float32")   # Band 4 = Red
    nir    = src.read(8).astype("float32")   # Band 8 = NIR
    nodata = src.nodata

valid = (red != nodata) & (nir != nodata)
ndvi  = np.where(valid, (nir - red) / (nir + red + 1e-9), np.nan)

plt.figure(figsize=(8, 6))
plt.imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(label="NDVI")
plt.title("NDVI — Sentinel-2")
plt.axis("off")
plt.tight_layout()
plt.show()

---
## NDVI Time Series — Mean per Image

In [ ]:
import glob, os
from datetime import datetime

image_files  = sorted(glob.glob(os.path.join(IMAGE_FOLDER, "*_nodata_clipped.tif")))
ndvi_series  = []

for img in image_files:
    with rasterio.open(img) as src:
        red = src.read(4).astype("float32")
        nir = src.read(8).astype("float32")
        nd  = src.nodata
    valid     = (red != nd) & (nir != nd)
    ndvi      = np.where(valid, (nir - red) / (nir + red + 1e-9), np.nan)
    parts     = os.path.basename(img).split("_")
    date      = datetime.strptime("_".join(parts[2:5]), "%Y_%m_%d")
    ndvi_series.append((date, float(np.nanmean(ndvi))))

ndvi_series.sort(key=lambda x: x[0])
dates, values = zip(*ndvi_series)

plt.figure(figsize=(10, 5))
plt.plot(dates, values, marker="o", color="green")
plt.title("NDVI Time Series (Full Image Mean)")
plt.xlabel("Date")
plt.ylabel("Mean NDVI")
plt.grid(True)
plt.tight_layout()
plt.show()

---
## NDVI Time Series — Per Crop Class

In [ ]:
with rasterio.open(CDL_PATH) as src:
    labels = src.read(1)

ndvi_per_class = {cid: [] for cid in CLASS_IDS}

for img in image_files:
    with rasterio.open(img) as src:
        assert src.shape == labels.shape, f"Shape mismatch: {img}"
        red = src.read(4).astype("float32")
        nir = src.read(8).astype("float32")
        nd  = src.nodata
    valid = (red != nd) & (nir != nd)
    ndvi  = np.where(valid, (nir - red) / (nir + red + 1e-9), np.nan)
    parts = os.path.basename(img).split("_")
    date  = datetime.strptime("_".join(parts[2:5]), "%Y_%m_%d")
    for cid in CLASS_IDS:
        mask = labels == cid
        if np.any(mask):
            ndvi_per_class[cid].append((date, float(np.nanmean(ndvi[mask]))))

plt.figure(figsize=(12, 7))
for cid in CLASS_IDS:
    series = sorted(ndvi_per_class[cid], key=lambda x: x[0])
    if series:
        d, v = zip(*series)
        plt.plot(d, v, marker="o", label=USDA_CDL_NAMES.get(cid, str(cid)))

plt.title("NDVI Time Series by Crop Class")
plt.xlabel("Date")
plt.ylabel("Mean NDVI")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---
## Label CSV Summary

In [ ]:
label_df = pd.read_csv(CSV_PATH)
label_df["count"] = pd.to_numeric(
    label_df["count"].astype(str).str.replace(",", ""), errors="coerce"
)
label_df = label_df.sort_values("count", ascending=False).reset_index(drop=True)

print(f"Total classes  : {len(label_df)}")
print(f"≥100k pixels   : {(label_df['count'] >= 100_000).sum()} classes")
label_df

In [ ]:
label_df.sort_values("count").plot(
    kind="barh", x="class_label", y="count",
    legend=False, figsize=(10, 12)
)
plt.title("CDL Class Distribution (from CSV)")
plt.xlabel("Pixel Count")
plt.tight_layout()
plt.show()

---
## Processed Dataset — Mask Sanity Check

In [ ]:
import glob

mask_dir   = "../data/processed/crop_mapping_multi_images_subset/masks"
unique_vals = set()

for f in glob.glob(mask_dir + "/*.tif"):
    with rasterio.open(f) as src:
        unique_vals |= set(np.unique(src.read(1)).tolist())

print("Unique class values across chip dataset:", sorted(unique_vals))